In [1]:
import numpy as np
import xarray as xr

In [2]:
def area_weighted_mean(values, area):
    values = np.asarray(values, dtype=float)
    area = np.asarray(area, dtype=float)

    valid = np.isfinite(values) & np.isfinite(area) & (area > 0)

    if valid.sum() == 0:
        return np.nan

    return np.sum(values[valid] * area[valid]) / np.sum(area[valid])

In [3]:
def regional_avg(n=None, data=None, path=''):
    """Calculate regional area-weighted values with the spatial resolution of 2.0°*2.0°."""

    latbnd = np.arange(-90 - n / 2, 90 + n, n)
    lat = int(180 / n + 1)
    lon = int(360 / n)

    if n == 0.25:
        lonbnd = np.arange(0 - n / 2, 360 + n / 2, n)
    else:
        lonbnd = np.arange(-180, 180 + n / 2, n)

    ds = xr.Dataset()

    ds.attrs['Source'] = 'PyGEMv1.1.0 developed by David Rounce (drounce@alaska.edu)'
    ds.attrs['Further developed by'] = 'Weilin Yang (weilinyang.yang@monash.edu)'
    ds.attrs['Code reviewed by'] = 'Wenchao Chu (peterchuwenchao@foxmail.com)'

    ds.coords['latitude'] = np.arange(-90, 90 + n, n)
    ds['latitude'].attrs['long_name'] = 'latitude'
    ds['latitude'].attrs['units'] = 'degrees_north'

    if n == 0.25:
        ds.coords['longitude'] = np.arange(0, 360, n)
        ds['longitude'].attrs['long_name'] = 'longitude'
        ds['longitude'].attrs['units'] = 'degrees_east'
    else:
        ds.coords['longitude'] = np.arange(-180 + n / 2, 180, n)
        ds['longitude'].attrs['long_name'] = 'longitude'
        ds['longitude'].attrs['units'] = 'degrees_east'

    ds['AAR_steady'] = (('latitude', 'longitude'), np.zeros([lat, lon]) * np.nan)
    ds['AAR_steady'].attrs['description'] = 'Area-weighted mean steady-state AAR'

    ds['AAR_mean'] = (('latitude', 'longitude'), np.zeros([lat, lon]) * np.nan)
    ds['AAR_mean'].attrs['description'] = 'Area-weighted mean AAR during the selected time period'

    ds['disequilibrium'] = (('latitude', 'longitude'), np.zeros([lat, lon]) * np.nan)
    ds['disequilibrium'].attrs['description'] = 'Area-weighted mean glacier-climate disequilibrium during the selected time period'

    for i in range(0, lat):
        bottom = latbnd[i]
        up = latbnd[i + 1]

        for j in range(0, lon):
            left = lonbnd[j]
            right = lonbnd[j + 1]

            find_id = np.where(
                (data['cenlat'] >= bottom)
                & (data['cenlat'] < up)
                & (data['cenlon'] >= left)
                & (data['cenlon'] < right)
            )[0]

            if len(find_id) != 0:
                area = data['rgi_area_km2'].values[find_id]

                ds['AAR_steady'].values[i, j] = area_weighted_mean(
                    data['AAR_steady'].values[find_id],
                    area,
                )
                ds['AAR_mean'].values[i, j] = area_weighted_mean(
                    data['AAR_mean'].values[find_id],
                    area,
                )
                ds['disequilibrium'].values[i, j] = area_weighted_mean(
                    data['disequilibrium'].values[find_id],
                    area,
                )

    path = path + 'PyGEM_glacier_stats_grid_' + str(n) + '.nc'
    enc_var = {'dtype': 'float32'}
    encoding = {v: enc_var for v in ds.data_vars}
    ds.to_netcdf(path, encoding=encoding)

In [4]:
path = '/g/data/rd53/wy2165/disequilibrium/pygem_oggm/';
fn = 'PyGEM_global_glacier_stats_median_a.nc';
data = xr.open_dataset(path+fn);
data = data.sel(experiment=data.gcm == 'era5')
data = data.squeeze('experiment')

n=2;
if n==0.25:
    find_id = np.where(data['cenlon'].values<-n/2);
    data['cenlon'].values[find_id] = data['cenlon'].values[find_id] + 360;
else:
    pass

regional_avg(n=n, data=data, path=path)